# NB-1: LoCoMo — CSAM vs Flat-RAG Baseline
**Publication blocker PB-02 + PB-03**

Runs CSAM (3-tier hierarchical memory) and a matched Flat-RAG baseline (L2-only)
on the LoCoMo long-conversation QA dataset, then computes the F1 delta with 95% CI.

**Time estimate:** ~20–40 min per model at max-conversations=5

**Free tiers that work:** Kaggle (30h/week GPU) · Google Colab (15h/week)

## Step 1 — Install dependencies & clone repo

In [ ]:
import os, subprocess, sys

# Install deps (quiet)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)
print('Deps installed')

# Clone / pull repo
REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'csam_project'))
print(f'Working dir: {os.getcwd()}')

## Step 2 — Set Groq API key
**Kaggle:** Notebook settings → Add-ons → Secrets → add `GROQ_API_KEY`

**Colab:** Left sidebar key icon → Secrets → add `GROQ_API_KEY`

In [ ]:
import os

# Try Kaggle secrets first, then Colab secrets, then env var
def load_api_key():
    # Kaggle
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('GROQ_API_KEY')
        if key:
            os.environ['GROQ_API_KEY'] = key
            return 'kaggle_secrets'
    except Exception:
        pass
    # Colab
    try:
        from google.colab import userdata
        key = userdata.get('GROQ_API_KEY')
        if key:
            os.environ['GROQ_API_KEY'] = key
            return 'colab_secrets'
    except Exception:
        pass
    # Already in env
    if os.environ.get('GROQ_API_KEY'):
        return 'env_var'
    raise RuntimeError('No GROQ_API_KEY found. Add it to Kaggle/Colab Secrets.')

source = load_api_key()
# Write .env so dotenv-loading scripts pick it up
with open('.env', 'w') as f:
    f.write(f"GROQ_API_KEY={os.environ['GROQ_API_KEY']}\n")
print(f'API key loaded from: {source}')

# Quick connectivity check
import requests
r = requests.get('https://api.groq.com/openai/v1/models',
    headers={'Authorization': f"Bearer {os.environ['GROQ_API_KEY']}"}, timeout=10)
models = [m['id'] for m in r.json().get('data', [])]
print(f'Groq connected — {len(models)} models available')
for m in ['llama-3.1-8b-instant', 'llama-3.3-70b-versatile']:
    status = '✓' if m in models else '✗'
    print(f'  {status} {m}')

## Step 3 — Configure run parameters
Edit these before running. Start with `MAX_CONV=3` to verify everything works (~5 min),
then set to `10` for publication results.

In [ ]:
# ── EDIT THESE ─────────────────────────────────────────────────
PROVIDER        = 'groq'
MODEL           = 'llama-3.1-8b-instant'   # fast; swap to llama-3.3-70b-versatile for full run
MAX_CONV        = 5       # 3 for a quick test, 10 for publication quality
QUESTIONS_CONV  = None    # None = all questions per conversation
SEED            = 42
DATASET         = 'csam_project/benchmarks/data/locomo10.json'
CHECKPOINT_DIR  = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
# ───────────────────────────────────────────────────────────────

import os
safe_model = MODEL.replace('/', '_')
OUT_CSAM     = f'csam_project/benchmarks/results_csam_{PROVIDER}_{safe_model}_s{SEED}.json'
OUT_BASELINE = f'csam_project/benchmarks/results_baseline_{PROVIDER}_{safe_model}_s{SEED}.json'
OUT_COMPARE  = f'csam_project/benchmarks/results_comparison_csam_vs_baseline_{safe_model}_s{SEED}.json'

print(f'Model:         {MODEL}')
print(f'Conversations: {MAX_CONV}')
print(f'Seed:          {SEED}')
print(f'CSAM output:   {OUT_CSAM}')
print(f'Base output:   {OUT_BASELINE}')

## Step 4 — Run CSAM benchmark (LoCoMo)
Runs the 3-tier CSAM system (L1+L2+L3 with consolidation-aware forgetting).

In [ ]:
import subprocess, sys, os

os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_multimodel',
    '--provider', PROVIDER,
    '--model', MODEL,
    '--dataset', DATASET,
    '--max-conversations', str(MAX_CONV),
    '--seed', str(SEED),
    '--checkpoint-dir', CHECKPOINT_DIR,
]
if QUESTIONS_CONV:
    cmd += ['--questions-per-conv', str(QUESTIONS_CONV)]

print('Running CSAM benchmark...')
print(' '.join(cmd[2:]))
result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    print('CSAM benchmark failed — check output above')
else:
    print(f'\nCSAM benchmark done. Output: {OUT_CSAM}')

## Step 5 — Run Flat-RAG Baseline (LoCoMo)

In [ ]:
cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_baseline_rag_hosted',
    '--provider', PROVIDER,
    '--model', MODEL,
    '--dataset', DATASET,
    '--max-conversations', str(MAX_CONV),
    '--seed', str(SEED),
    '--checkpoint-dir', CHECKPOINT_DIR,
]
if QUESTIONS_CONV:
    cmd += ['--questions-per-conv', str(QUESTIONS_CONV)]

print('Running Flat-RAG Baseline...')
result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    print('Baseline benchmark failed')
else:
    print(f'\nBaseline done. Output: {OUT_BASELINE}')

## Step 6 — Compare CSAM vs Baseline + Print Results

In [ ]:
import json, os

# Run comparison script
cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.compare_csam_vs_baseline',
    '--csam', OUT_CSAM,
    '--baseline', OUT_BASELINE,
    '--n-bootstrap', '2000',
]
result = subprocess.run(cmd, capture_output=False, text=True)

# Pretty-print the comparison JSON
if os.path.exists(OUT_COMPARE):
    with open(OUT_COMPARE) as f:
        cmp = json.load(f)
    print('\n=== FINAL COMPARISON SUMMARY ===')
    print(f'Model:          {cmp["model_id"]}')
    print(f'Conversations:  {cmp["num_conversations"]}')
    print(f'CSAM Macro F1:  {cmp["csam_macro_f1"]:.4f}')
    print(f'Base Macro F1:  {cmp["baseline_macro_f1"]:.4f}')
    print(f'Delta (CSAM-B): {cmp["macro_f1_delta"]:+.4f}')
    lo, hi = cmp['delta_ci_95']
    print(f'95% CI on Δ:   [{lo:+.4f}, {hi:+.4f}]')
    print(f'Significance:   {cmp["statistical_significance"]}')
else:
    print('Comparison file not found — check if both benchmarks completed.')

## Step 7 — Save / Download results

In [ ]:
import shutil, os

files_to_save = [OUT_CSAM, OUT_BASELINE, OUT_COMPARE]

# Kaggle: copy to /kaggle/working (auto-saved as output)
if os.path.exists('/kaggle'):
    for fp in files_to_save:
        if os.path.exists(fp):
            dest = os.path.join('/kaggle/working', os.path.basename(fp))
            shutil.copy(fp, dest)
            print(f'Saved to Kaggle output: {dest}')

# Colab: trigger browser download
else:
    try:
        from google.colab import files
        for fp in files_to_save:
            if os.path.exists(fp):
                files.download(fp)
                print(f'Download triggered: {fp}')
    except ImportError:
        print('Not on Colab — files are at:', [fp for fp in files_to_save if os.path.exists(fp)])